# Modes report for one emotion, end to end (Colab, L4)

`scripts/pipeline.py` runs harvest -> modes -> judge -> report, with a checkpoint after each stage. Re-running it skips finished stages.

1. **Runtime > Change runtime type > L4 GPU** before running anything (changing it later restarts the VM).
2. Optional Colab secrets (key icon on the left): `OPENROUTER_API_KEY` to run the judge and report here too; `AWS_ACCESS_KEY_ID` / `AWS_SECRET_ACCESS_KEY` for S3 checkpoints that survive the VM.
3. Set `CONFIG` in the checkout cell (e.g. `configs/harvest_angry_l25.yaml`), and `S3` if you use S3.
4. Run the cells in order; re-run the status cell. Harvest ~12 min, modes ~25-30 min, judge ~15 min, report ~1 min.
5. The download cell zips `results/<run_id>` (report, CSV, judgements, modes; ~30 MB).

Without an OpenRouter key here, the run stops after modes; download the results, then on your machine run
`uv run python scripts/pipeline.py --config <CONFIG>`: it skips harvest and modes and finishes judge and report.
(With S3, use the same `--pipeline.s3-prefix` on both machines and nothing needs downloading.)

In [ ]:
# setup (once per VM)
import os
from google.colab import userdata
for name in ("OPENROUTER_API_KEY", "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "HF_TOKEN"):
    try:
        os.environ[name] = userdata.get(name)
        print("secret", name, "set")
    except Exception:
        print("secret", name, "not set")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")
import torch
print("torch before install:", torch.__version__)
!pip -q install uv awscli
!test -d /content/emotion-neurons || git clone -q https://github.com/narang99/emotion-neurons.git /content/emotion-neurons
%cd /content/emotion-neurons
!uv pip install --system -q --no-deps -e .
!uv pip install --system -q pydantic-settings einops jaxtyping
!python -c "import torch; print('torch', torch.__version__, torch.cuda.get_device_name(0)); assert torch.cuda.is_available() and '+cu' in torch.__version__"

In [ ]:
# checkout + config
BRANCH = "stack/harvest-happy"
CONFIG = "configs/harvest_angry_l25.yaml"
S3 = None   # e.g. "s3://your-bucket/emotion-neurons": checkpoints there survive the VM

%cd /content/emotion-neurons
!git fetch -q origin
!git checkout -q {BRANCH}
!git pull -q --ff-only
!uv pip install --system -q --no-deps -e .
branch = !git rev-parse --abbrev-ref HEAD
assert branch[0] == BRANCH, branch
!git log --oneline -1
import os, yaml
RUN_ID = yaml.safe_load(open(CONFIG))["run_id"]
UNTIL = "report" if os.environ.get("OPENROUTER_API_KEY") else "modes"
print("config", CONFIG, "| run_id", RUN_ID, "| runs until", UNTIL, "| S3", S3)

In [ ]:
# run the pipeline in the background (safe to re-run: finished stages are skipped)
import subprocess
alive = subprocess.run(["pgrep", "-f", "scripts/pipeline.py"], capture_output=True, text=True).stdout.split()
if alive:
    print("already running, pids", alive)
else:
    cmd = f"PYTHONUNBUFFERED=1 python scripts/pipeline.py --config {CONFIG} --pipeline.until {UNTIL}"
    if S3:
        cmd += f" --pipeline.s3-prefix {S3}"
    proc = subprocess.Popen(cmd + "; echo EXIT $?", shell=True, cwd="/content/emotion-neurons", stdin=subprocess.DEVNULL,
                            stdout=open("/content/pipeline.log", "a"), stderr=subprocess.STDOUT, start_new_session=True)
    print("started, pid", proc.pid, "|", cmd)

In [ ]:
# status: wait for "[pipeline] stopped after ..." and "EXIT 0"
!tr '\r' '\n' < /content/pipeline.log | grep -E "^\[|Error|Traceback|EXIT" | tail -15
import subprocess
print(subprocess.run(["pgrep", "-af", "scripts/pipeline.py"], capture_output=True, text=True).stdout or "no run alive")

In [ ]:
# download the results (report.html, neurons.csv, judgements, modes, harvest.json, stage markers, the log)
assert "EXIT 0" in open("/content/pipeline.log").read(), "the pipeline has not finished cleanly"
!cp /content/pipeline.log /content/emotion-neurons/results/{RUN_ID}/
!cd /content/emotion-neurons/results && zip -q -r /content/{RUN_ID}_results.zip {RUN_ID}
from google.colab import files
files.download(f"/content/{RUN_ID}_results.zip")

In [ ]:
# optional: keep the 2-3 GB corpus pass (runs/<run_id>) on your Google Drive (only needed to redo modes)
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/emotion-neurons && cp -r /content/emotion-neurons/runs/{RUN_ID} /content/drive/MyDrive/emotion-neurons/

In [ ]:
# release the VM when everything is saved (credits burn on wall-clock time)
from google.colab import runtime
runtime.unassign()